In [ ]:
import meshio as mio
import h5py
import numpy as np
import pyvista as pv
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
import scipy as sp

pv.set_jupyter_backend('trame')


In [ ]:
def to_pyvista_mesh(V, F = None):
    if F is None:
        return pv.PolyData(V)
    if F.shape[1] == 3:
        return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)
    elif F.shape[1] == 4:
        return pv.UnstructuredGrid({pv.CellType.TETRA: F}, V)



In [ ]:
# path = "/Users/teseo/Downloads/Embryogram test/new/0in-analysis-07_09T15_10-analysis.hdf5"
path = "/Users/teseo/Downloads/Embryogram test/20250510_tracking-analysis-07_19T18_47-analysis.hdf5"

In [ ]:
hdf5_file = h5py.File(path, "r")
V, T = hdf5_file["mesh/v"][:].astype(float), hdf5_file["mesh/t"][:].astype(np.int32)

top = hdf5_file["bc/top"][:].astype(np.int32)
bottom = hdf5_file["bc/bottom"][:].astype(np.int32)
middle = hdf5_file["bc/middle"][:].astype(np.int32)

In [ ]:
centers = hdf5_file["bc_func/centers"][:].astype(float)

In [ ]:
nk = len(hdf5_file["bc_func"].keys())-2

disps = []

for i in range(nk):
    disps.append(hdf5_file[f"bc_func/disp{i+1}"][:].astype(float))

In [ ]:
# m=to_pyvista_mesh(V, T)
# # m=m.explode(0.5)
# plt = pv.Plotter()
# plt.add_mesh(m, show_edges=True, style='wireframe', line_width=1.0)
# plt.add_mesh(to_pyvista_mesh(V[top]), color='red', point_size=10, render_points_as_spheres=True, name='top')
# plt.add_mesh(to_pyvista_mesh(V[bottom]), color='green', point_size=10, render_points_as_spheres=True, name='bottom')
# plt.add_mesh(to_pyvista_mesh(V[middle]), color='blue', point_size=10, render_points_as_spheres=True, name='middle')
# plt.show()

In [ ]:
# m=m.explode(0.5)
plt = pv.Plotter()
plt.add_mesh(to_pyvista_mesh(V[middle]), color='blue', point_size=8, render_points_as_spheres=True, name='middle')
plt.add_mesh(to_pyvista_mesh(centers), color='red', point_size=7, render_points_as_spheres=True, name='middle')
plt.show()

In [ ]:
pl = pv.Plotter()
pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

nd = disps[0].shape[0]
lines = np.hstack([[2, i, nd+i] for i in range(nd)])

vertices = np.vstack([centers, centers + disps[0]])
ll = pv.PolyData(vertices, lines=lines)
pl.add_mesh(ll, name='line')

def callback(x):
    vertices = np.vstack([centers, centers + disps[x]])
    ll = pv.PolyData(vertices, lines=lines)
    pl.add_mesh(ll, name='line')
    pl.update()

pl.show()
interact(callback, x=(0, len(disps)-1, 1))



$$
P=(p_0^0|\dots|p_0^n)\qquad
Q=(p_1^0|\dots|p_1^n)
$$

$$
\min_{M, t} E=\sum_i \|P_i-MQ_i - t\|^2
$$

$$
E = \sum_i (
P_i^T P_i
-2P_i^T M Q_i
-2P_i^T t
+ 2Q_i^T M^T t
+Q_i^T M^T M Q_i
+  t^T t)
$$
$$
=
tr(P^T P)
-2tr(P^T M Q)
-2P^T t o
+2Q^T M^T t o
+tr(Q^T M^T M Q)
+  n t^T t
$$



$$
\nabla_M E = 
-2 P Q^T
+ 2  ot Q^T
+2MQ  Q^T
$$

$$
\nabla_t E = 
-2P o^T
+ 2M Q   o^T
+ 2t o  o^T
$$

In [ ]:
import sympy as sp 

n = sp.Symbol('n', integer=True)
P = sp.MatrixSymbol('P', 3, 100)
Q = sp.MatrixSymbol('Q', 3, 100)
one = sp.MatrixSymbol('o', 1, 100)

M = sp.MatrixSymbol('M', 3, 3)
t = sp.MatrixSymbol('t', 3, 1)

Es = sp.trace(P.T * P)-2*sp.trace(P.T*M*Q) -sp.trace(2*one*P.T*t)+ sp.trace(2*one*Q.T*M.T*t)+sp.trace(Q.T*M.T*M*Q) + sp.trace(t.T*t)*n


Es = sp.simplify(sp.expand(Es))

Es

In [ ]:
sp.diff(Es, t)

In [ ]:
# Es = sp.simplify(sp.expand(sp.trace((P-M*Q-t*one).T*(P-M*Q-t*one))))
# Es

In [ ]:
o=one

E = P.T *P-2*P.T*M*Q-2*P.T *t*o + 2*Q.T* M.T* t*o+Q.T*M.T*M*Q + o.T* t.T* t* o
E-Es
# E

In [ ]:
sp.diff(E, M)

In [ ]:
import sympy as sp 


t = sp.MatrixSymbol('t', 3, 1)

X = sp.MatrixSymbol('x', 3, 1)
M = sp.MatrixSymbol('M', 3, 3)
PP = sp.MatrixSymbol('P', 3, 3)
K = sp.MatrixSymbol('K', 3, 1)



t1=X*t.T

t2 = M*PP


t3 = M.T * K
t3[0,0]


In [ ]:
def align(centers, disps, index):
    p0 = centers.T
    p1 = (centers + disps[index]).T
    one = np.ones((1, p0.shape[1]))

    n = one.shape[1]


    X = p1 @ one.T
    PP = p1 @ p1.T
    K = p1 @ one.T
    
    system = np.zeros((9+3, 9+3))
    rhs = np.zeros((9+3, ))

    rhs[:9] = (p0 @ p1.T).flatten()
    rhs[9:] = (p0 @ one.T).flatten()


    system[0,:3] = PP[:,0]
    system[1,:3] = PP[:,1]
    system[2,:3] = PP[:,2]

    system[3,3:6] = PP[:,0]
    system[4,3:6] = PP[:,1]
    system[5,3:6] = PP[:,2]

    system[6,6:9] = PP[:,0]
    system[7,6:9] = PP[:,1]
    system[8,6:9] = PP[:,2]


    system[0, 9] = X[0,0]
    system[1, 10] = X[0,0]
    system[2, 11] = X[0,0]

    system[0, 9] = X[1,0]
    system[1, 10] = X[1,0]
    system[2, 11] = X[1,0]

    system[0, 9] = X[2,0]
    system[1, 10] = X[2,0]
    system[2, 11] = X[2,0]

    system[9,:3] = K[:,0]
    system[9,3:6] = K[:,0]
    system[9,6:9] = K[:,0]

    system[9, 9] = n
    system[10, 10] = n
    system[11, 11] = n

    sol = np.linalg.solve(system, rhs)

    t = sol[9:12]

    return sol[:9].reshape((3, 3)), np.zeros(3,) #t.reshape((3, ))

M, t = align(centers, disps, 1)

print(M)
print(t)


In [ ]:
adf = np.arange(12)

adf[9:]

In [ ]:
import scipy.optimize as opt


def align_opt(centers, disps, index):
    P = centers.T
    # Q = (centers + disps[index]).T
    Q = P.copy()+ np.array([1, 2, 3]).reshape((3, 1))  # Example adjustment

    E = lambda x: np.sum((P - (x[:9].reshape(3, 3) @ Q + x[9:].reshape((3, 1))))**2)

    M = np.eye(3)
    t = np.zeros(3,)
    x0 = np.concatenate((M.flatten(), t))

    xs=opt.fmin(func=E, x0=x0, disp=True)

    # print(E(x0))
    # print(np.sum(disps[index]**2))

    Ms = xs[:9].reshape((3, 3))
    ts = xs[9:].reshape((3, ))

    return Ms, ts


align_opt(centers, disps, 1)


In [ ]:
def umeyama(P, Q):
    assert P.shape == Q.shape
    n, dim = P.shape

    centeredP = P - P.mean(axis=0)
    centeredQ = Q - Q.mean(axis=0)

    C = np.dot(np.transpose(centeredP), centeredQ) / n

    V, S, W = np.linalg.svd(C)
    d = (np.linalg.det(V) * np.linalg.det(W)) < 0.0

    if d:
        S[-1] = -S[-1]
        V[:, -1] = -V[:, -1]

    R = np.dot(V, W)

    varP = np.var(P, axis=0).sum()
    c = 1/varP * np.sum(S) # scale factor

    t = Q.mean(axis=0) - P.mean(axis=0).dot(c*R)

    return c, R, t

def align_1(centers, disps, index):
    P = centers.T
    Q = (centers + disps[index]).T
    # Q = P.copy()+ np.array([1, 2, 3]).reshape((3, 1))  # Example adjustment
    # Q = 100*P+ np.array([1, 2, 3]).reshape((3, 1))
    # print(P.shape, Q.shape)
    c, R, t = umeyama(P.T, Q.T)

    # return c, R, t
    return c*R, t

align_1(centers, disps, 1)


In [ ]:
def icp(centers, d):
    p0 = centers
    p1 = centers + d
    mu0 = np.mean(p0, axis=0)
    mu1 = np.mean(p1, axis=0)

    p0 -= mu0
    p1 -= mu1
    t = mu0 - mu1

    cov = p1.T @ p0
    U, s, Ut = sp.linalg.svd(cov)
    R = Ut @ U.T

    return R, t

In [ ]:
def compute_align_disp(centers, disps, index):
    # M, t = align_opt(centers, disps, index)
    # M, t = align(centers, disps, index)
    # M, t = icp(centers, disps[index])
    M, t = align_1(centers, disps, index)

    tmp = centers + disps[index]
    tmp1 = (M @ tmp.T).T + t

    return tmp1-centers, M, t

compute_align_disp(centers, disps, 1)

In [ ]:
new_disps = []
Ms = []
ts = []
for i in range(len(disps)):
    d, M, t = compute_align_disp(centers, disps, i)
    new_disps.append(d)
    Ms.append(M)
    ts.append(t)

In [ ]:
pl = pv.Plotter()
pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

nd = disps[0].shape[0]
lines = np.hstack([[2, i, nd+i] for i in range(nd)])

vertices = np.vstack([centers, centers + disps[0]])
ll = pv.PolyData(vertices, lines=lines)
pl.add_mesh(ll, name='line', color='red')

vertices = np.vstack([centers, centers + new_disps[0]])
ll = pv.PolyData(vertices, lines=lines)
pl.add_mesh(ll, name='linea', color='green')

def callback(x):
    vertices = np.vstack([centers, centers + disps[x]])
    ll = pv.PolyData(vertices, lines=lines)
    pl.add_mesh(ll, name='line', color='red')

    vertices = np.vstack([centers, centers + new_disps[x]])
    ll = pv.PolyData(vertices, lines=lines)
    pl.add_mesh(ll, name='linea', color='green')
    pl.update()

    print(Ms[x])
    print(ts[x])

pl.show()
interact(callback, x=(0, len(disps)-1, 1), continuous_update=False)





In [ ]:
f=206
plt.hist(np.linalg.norm(np.array(disps[f]-new_disps[f]), axis=1), bins=50)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

data = [np.linalg.det(M) for M in Ms]
plt.plot(data)
plt.xlabel('time')
plt.ylabel('scale')
plt.show()

In [ ]:
data = [t[0] for t in ts]
plt.plot(data, label='x')
data = [t[1] for t in ts]
plt.plot(data, label='y')
data = [t[2] for t in ts]
plt.plot(data, label='z')
plt.legend()
plt.xlabel('time')
plt.ylabel('translation')
plt.show()

In [ ]:
from __future__ import print_function
import cv2 as cv
import numpy as np
import argparse
 
 
parser = argparse.ArgumentParser(description='Code for Affine Transformations tutorial.')
parser.add_argument('--input', help='Path to input image.', default='lena.jpg')
args = parser.parse_args()
 
src = cv.imread(cv.samples.findFile(args.input))
if src is None:
    print('Could not open or find the image:', args.input)
    exit(0)
 
 
 
srcTri = np.array( [[0, 0], [src.shape[1] - 1, 0], [0, src.shape[0] - 1]] ).astype(np.float32)
dstTri = np.array( [[0, src.shape[1]*0.33], [src.shape[1]*0.85, src.shape[0]*0.25], [src.shape[1]*0.15, src.shape[0]*0.7]] ).astype(np.float32)
 
 
 
warp_mat = cv.getAffineTransform(srcTri, dstTri)
 
 
 
warp_dst = cv.warpAffine(src, warp_mat, (src.shape[1], src.shape[0]))
 
 
# Rotating the image after Warp
 
 
center = (warp_dst.shape[1]//2, warp_dst.shape[0]//2)
angle = -50
scale = 0.6
 
 
 
rot_mat = cv.getRotationMatrix2D( center, angle, scale )
 
 
 
warp_rotate_dst = cv.warpAffine(warp_dst, rot_mat, (warp_dst.shape[1], warp_dst.shape[0]))
 
 
 
cv.imshow('Source image', src)
cv.imshow('Warp', warp_dst)
cv.imshow('Warp + Rotate', warp_rotate_dst)
 
 
 
cv.waitKey()